# Fine-tuning Ember-NNUE with nnue-pytorch

This notebook fine-tunes the **external network for the Ember engine** in the binary container
format V2 (magic `0x6A448AFA`):

| Parameter           | Value                                     |
|---------------------|-------------------------------------------|
| `Full_Threats`      | 60,720 inputs, i8 weights                 |
| `HalfKAv2_hm^` (PSQ)| 22,528 inputs, i16 weights                |
| `L1` (hidden)       | 1024 (SCReLU)                             |
| `Affine` stacks     | 8 buckets × (32 → 32 → 1), SCCReLU        |
| PSQT buckets        | 8                                         |
| ARCH_HASH           | `0x0256acdf`                              |
| FT_HEADER_HASH      | `0x6165ddc9`                              |
| STACK_HASH          | `0x63337116`                              |

Dataset: [farseerT74](https://huggingface.co/datasets/official-stockfish/master-binpacks/blob/main/farseerT74.binpack)
(~20 GB). The notebook downloads it, installs dependencies,
and builds the C++ data loader.

**Run order**
1. Runtime → Change runtime type → **GPU** (T4 / A100 / L4).
2. Run all cells in order. Downloading the data takes a while.
3. One superbatch = 100,000,000 positions = 6104 batches × 16384. By default
   300 superbatches with auto-save every **3**.
4. If the session dies, restart from cell 11 — training resumes from the last
   checkpoint (checkpoints live on Google Drive).


## 0. Environment and GPU


In [ ]:
!nvidia-smi
import torch, sys
print("Python", sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. nnue-pytorch (pinned commit)

Uses commit `a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82` — the last one before
`PP_3Wide` was added. It has `Full_Threats` = 60,720 inputs and the default
`Full_Threats+HalfKAv2_hm^`, i.e. exactly the architecture Ember expects.


In [ ]:
%%bash
set -e
cd /content
if [ ! -d /content/nnue-pytorch/.git ]; then
  echo "Cloning nnue-pytorch..."
  git clone https://github.com/official-stockfish/nnue-pytorch /content/nnue-pytorch
fi
cd /content/nnue-pytorch
git fetch --quiet --all 2>/dev/null || true
git checkout -q a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82
git rev-parse HEAD
echo "OK"


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq >/dev/null 2>&1
apt-get install -y -qq cmake g++ >/dev/null 2>&1
pip install -q -r /content/nnue-pytorch/requirements.txt
pip uninstall -y -q tensorflow tensorflow-cpu tf-nightly tf-keras 2>/dev/null || true
python -c "import torch, tyro, lightning; print('torch', torch.__version__, '| tyro OK | lightning', lightning.__version__)"


### Building the C++ data loader

Without `libtraining_data_loader.so` training will not start — it is linked via
ctypes from `./build/`, so training must be launched from the `/content/nnue-pytorch` directory.


In [ ]:
%%bash
set -e
cd /content/nnue-pytorch
cmake -S data_loader/cpp -B build -DCMAKE_BUILD_TYPE=Release > /tmp/cmake_cfg.log 2>&1
cmake --build build -j"$(nproc)" > /tmp/cmake_build.log 2>&1
ls -la build/libtraining_data_loader* build/training_data_loader* 2>/dev/null | head
cd /content/nnue-pytorch && python -c "import data_loader; print('data_loader OK')"


## 2. Dataset `farseerT74.binpack` (~20 GB)

The file is downloaded to `/content/data` (runtime disk).

In [ ]:
%%bash
set -e
URL="https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/farseerT74.binpack?download=true"
DATASET="/content/data/farseerT74.binpack"

mkdir -p /content/data
echo "Downloading the file..."
curl -L -o "$DATASET" "$URL"
echo "Download complete!"
ls -lh "$DATASET"

## 3. Training

**A superbatch is one epoch.** Parameters are set in the cell below. Training writes
Lightning checkpoints to `OUT_ROOT` on Google Drive; on a re-run it automatically picks up
the latest `checkpoints/last.ckpt` (`--resume-from-checkpoint`).

Defaults: 300 superbatches, auto-save every **3**, batch 16384, epoch = 100M positions,
rangerlite optimiser (the standard SF pipeline), WDL-remapped loss as in the config.


In [ ]:
import os

os.environ['NNUE_REPO']   = '/content/nnue-pytorch'
os.environ['DATASET']     = '/content/data/farseerT74.binpack'
os.environ['DRIVE_DIR']   = '/content/drive/MyDrive/Ember_Networks'
os.environ['RUN_NAME']    = 'ember_ft_hm_2026'
os.environ['MAX_EPOCHS']  = '300'
os.environ['SAVE_EVERY']  = '3'
os.environ['BATCH_SIZE']  = '16384'
os.environ['EPOCH_SIZE']  = '100000000'
os.environ['NUM_WORKERS'] = '4'
os.environ['SMOKE']       = 'False'

print("config:", {k: os.environ[k] for k in ['NNUE_REPO','DATASET','DRIVE_DIR','RUN_NAME','MAX_EPOCHS','SAVE_EVERY','BATCH_SIZE','EPOCH_SIZE','NUM_WORKERS']})


In [ ]:
import os, sys, glob, signal, subprocess

os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
os.environ["CUDNN_DETERMINISTIC"] = "0"
os.environ["CUDNN_BENCHMARK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PL_TORCH_BACKEND"] = "torch"
os.environ["LIGHTNING_PRECISION"] = "16-mixed"

NNUE_REPO = os.environ['NNUE_REPO']
DATASET   = os.environ['DATASET']
DRIVE_DIR = os.environ['DRIVE_DIR']
RUN_NAME  = os.environ['RUN_NAME']
OUT_ROOT  = f"{DRIVE_DIR}/{RUN_NAME}"

max_epochs   = int(os.environ['MAX_EPOCHS'])
save_every   = int(os.environ['SAVE_EVERY'])
batch_size   = int(os.environ['BATCH_SIZE'])
epoch_size   = int(os.environ['EPOCH_SIZE'])
num_workers  = int(os.environ['NUM_WORKERS'])
smoke        = os.environ.get('SMOKE', 'False').strip().lower() in ('1','true','yes')

assert os.path.exists(DATASET), "dataset not found, run download cell first"

if smoke:
    max_epochs = 2
    save_every = 1
    epoch_size = min(epoch_size, 500_000)
    num_workers = 2
    print(">>> SMOKE TEST: 2 superbatches of", epoch_size, "positions")

os.makedirs(OUT_ROOT, exist_ok=True)

ckpts = sorted(glob.glob(f"{OUT_ROOT}/**/checkpoints/last.ckpt", recursive=True),
               key=os.path.getmtime)
resume = ckpts[-1] if ckpts else None

cmd = [
    sys.executable, "train.py", DATASET,
    "--features", "Full_Threats+HalfKAv2_hm^",
    "--l1", "1024", "--l2", "32", "--l3", "32",
    "--batch-size", str(batch_size),
    "--epoch-size", str(epoch_size),
    "--max-epochs", str(max_epochs),
    "--network-save-period", str(save_every),
    "--save-top-k", "-1",
    "--num-workers", str(num_workers),
    "--default-root-dir", OUT_ROOT,
    "--accelerator", "cuda",
]
if resume:
    cmd += ["--resume-from-checkpoint", resume]
    print(">>> resume from", resume)
print("$", " ".join(cmd), "\n")

process = subprocess.Popen(
    cmd,
    cwd=NNUE_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
    env=os.environ.copy(),
    start_new_session=True,
)

def terminate_process_group(process):
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()

try:
    for line in process.stdout:
        print(line, end='')
except KeyboardInterrupt:
    terminate_process_group(process)
    raise

return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)